<a href="https://colab.research.google.com/github/dnwabeze/Token-BO/blob/main/Running-Ollama-on-Google-Colab-Through-Pinggy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install pciutils and Ollama
!sudo apt-get install -y pciutils
!curl https://ollama.ai/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 33 not upgraded.
Need to get 381 kB of archives.
After this operation, 1,759 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Fetched 381 kB in 2s (196 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 3.)
debconf: falling back to frontend: R

In [2]:
# Install pinggy
!pip install pinggy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 113.4 MB/s eta 0:00:00


In [15]:
# Start the Ollama server in a subprocess by downloading the official binary directly
import subprocess
import os

def start_ollama_server():
    ollama_path = '/usr/local/bin/ollama'

    # Remove the path if it fails standard execution to ensure no corrupted files persist
    if os.path.exists(ollama_path):
        try:
            subprocess.run([ollama_path, '--version'], capture_output=True)
        except OSError:
            print("⚠️ Corrupted binary detected. Removing for a clean installation...")
            subprocess.run(["sudo", "rm", "-f", ollama_path])

    # Run the official installer cleanly with OLLAMA_INSTALL_ONLY=1 after installing zstd
    if not os.path.exists(ollama_path):
        print("📦 Installing missing dependency 'zstd'...")
        subprocess.run(["sudo", "apt-get", "update", "-y"], capture_output=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "zstd"], capture_output=True)

        print("📥 Installing Ollama officially...")
        result = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sudo OLLAMA_INSTALL_ONLY=1 sh",
            shell=True,
            capture_output=True,
            text=True
        )
        if result.returncode != 0:
            print("❌ Installer returned non-zero exit code!")
            print("STDOUT:", result.stdout)
            print("STDERR:", result.stderr)

    if os.path.exists(ollama_path):
        subprocess.Popen([ollama_path, 'serve'])
        print("🚀 Ollama server launched successfully!")
    else:
        print("❌ Error: Ollama installation failed.")

start_ollama_server()

📦 Installing missing dependency 'zstd'...
📥 Installing Ollama officially...
🚀 Ollama server launched successfully!


In [16]:
# Check if Ollama is listening on port 11434
def check_ollama_port(port='11434'):
    try:
        output = subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'],
                              capture_output=True, text=True).stdout
        if f":{port} (LISTEN)" in output:
            print(f"✅ Ollama is actively listening on port {port}")
        else:
            print(f"⚠️ Ollama not detected on port {port}")
    except Exception as e:
        print(f"❌ Error checking port: {e}")
check_ollama_port()

✅ Ollama is actively listening on port 11434


In [17]:
# Start a pinggy tunnel to expose the Ollama API
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:11434",
    headermodification=["u:Host:localhost:11434"]
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

Tunnel1 started - URLs: ['https://efnau-34-126-177-35.run.pinggy-free.link', 'https://gqkqq-34-126-177-35.free.pinggy.net']


In [18]:
# Pull the llama3.2:1b model
!ollama pull llama3.2:1b

In [23]:
# Make a request directly to the local Ollama API
import requests
import json

try:
    # Since we are running in the same Colab environment, query localhost directly to bypass public tunnel checks
    local_url = "http://localhost:11434"
    print(f"🔗 Sending request directly to local Ollama server: {local_url}\n")

    response = requests.post(
        f"{local_url}/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": "Hello, how are you?",
            "stream": False
        }
    )

    response.raise_for_status()
    print(response.json()["response"])
except Exception as e:
    print(f"❌ Local request failed: {e}")

🔗 Sending request directly to local Ollama server: http://localhost:11434

I'm doing well, thank you for asking. Is there anything I can help you with or would you like to talk about something in particular?


In [25]:
# Install Open WebUI by ignoring strict Python version constraints
!pip install open-webui --ignore-requires-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [26]:
# Start a pinggy tunnel for Open WebUI
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8000",
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

Tunnel1 started - URLs: ['https://lnmic-34-126-177-35.free.pinggy.net', 'https://qeodd-34-126-177-35.run.pinggy-free.link']


In [ ]:
# Start the Open WebUI server on port 8000
!open-webui serve --port 8000

Loading WEBUI_SECRET_KEY from file, not provided as an environment variable.
Generating a new secret key and saving it to /content/.webui_secret_key
Loading WEBUI_SECRET_KEY from /content/.webui_secret_key
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 7e5b5dc7342b
INFO  [alembic.runtime.migration] Running upgrade 7e5b5dc7342b -> ca81bd47c050, Add config table.
INFO  [alembic.runtime.migration] Running upgrade ca81bd47c050 -> c0fbf31ca0db, Update file table
INFO  [alembic.runtime.migration] Running upgrade c0fbf31ca0db -> 6a39f3d8e55c, Add knowledge table
Creating knowledge table
Migrating data from document table to knowledge table
INFO  [alembic.runtime.migration] Running upgrade 6a39f3d8e55c -> 242a2047eae0, Update chat table
Converting 'chat' column to JSON
Renaming 'chat' column to 'old_chat'
Adding new 'chat' column of type JSON
Dropping 'old_chat